# JWCM v2: Jacobian-Weighted Criticality Merging

## Overview

This notebook implements **JWCM v2**, a principled merging method for RLVR-trained models.

### Core Idea
For each parameter $\theta_j$, compute an importance score $S_j^\tau$ that measures
how much $\theta_j$ contributes to **critical token probability changes** ($\Delta \log p$).

### Attribution Score
$$S_j^\tau = \sum_{(x, y_t) \in \mathcal{C}_\tau} \left| \frac{\partial \log p(y_t)}{\partial \theta_j} \cdot \Delta\theta_j^\tau \right|$$

### Merge Formula (N-task scalable)
$$\theta_j^{\text{merge}} = \theta_j^{\text{base}} + \frac{\sum_\tau S_j^\tau \cdot \Delta\theta_j^\tau}{\sum_\tau S_j^\tau + \epsilon}$$

### Sparsification
$$\theta_j^{\text{merge}} = \theta_j^{\text{base}} + \mathbb{1}[\max_\tau S_j^\tau > \kappa] \cdot \frac{\sum_\tau S_j^\tau \cdot \Delta\theta_j^\tau}{\sum_\tau S_j^\tau + \epsilon}$$

### Outputs
- `merged_model/`: JWCM v2 merged model (soft attribution)
- `merged_model_sparse/`: JWCM v2 + sparsification (top 20% parameters only)


In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================================================
# Configuration
# ============================================================================
BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface"
MATH_MODEL_PATH = "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface"

IF_DATA_PATH = "/mnt/ddn/vuvlm/geeho/datasets/Nemotron-Cascade-RL-IF/ifrl_final_release.parquet"
MATH_DATA_PATH = "/mnt/ddn/vuvlm/geeho/datasets/Nemotron-Cascade-RL-Math/math_verl_ready_MERGED_SYSTEM.parquet"

# Validation set size: 512 per benchmark
VAL_SAMPLES_PER_TASK = 512

# Generation settings
MAX_NEW_TOKENS = 512          # Max response length for generation
GENERATION_BATCH_SIZE = 16    # Batch size for greedy decoding

# Attribution settings
CRITICAL_PERCENTILE = 90      # Top 10% of |delta_log_p| are critical tokens
SPARSE_KEEP_RATIO = 0.20      # For v2+sparse: keep top 20% of parameters

# Reproducibility
SEED = 42
DTYPE = torch.float16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style="whitegrid", context="talk")


In [ ]:
# ============================================================================
# Model and Data Utilities
# ============================================================================

def load_model(model_id_or_path: str, dtype=torch.float16, device: str = "cpu"):
    """Load a causal LM model.

    Args:
        model_id_or_path: HuggingFace model ID or local path.
        dtype: Weight dtype.
        device: Target device ("cpu" or "cuda").

    Returns:
        AutoModelForCausalLM instance.
    """
    model = AutoModelForCausalLM.from_pretrained(
        str(model_id_or_path),
        torch_dtype=dtype,
        device_map=device,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    return model


def load_tokenizer(model_id_or_path: str = BASE_MODEL_ID):
    """Load tokenizer with pad token configured for batch generation.

    Qwen3 models may not have an explicit pad token. We set it to eos_token
    so that left-padded batches work correctly during generation.

    Args:
        model_id_or_path: HuggingFace model ID or local path.

    Returns:
        AutoTokenizer instance.
    """
    tokenizer = AutoTokenizer.from_pretrained(
        str(model_id_or_path),
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    return tokenizer


def sample_validation_data(parquet_path: str, n: int, seed: int = 42) -> pd.DataFrame:
    """Sample n rows from a parquet file for validation.

    Args:
        parquet_path: Path to the parquet dataset.
        n: Number of samples to draw.
        seed: Random seed for reproducibility.

    Returns:
        Sampled DataFrame with reset index.
    """
    df = pd.read_parquet(parquet_path)
    if len(df) < n:
        print(f"Warning: dataset has only {len(df)} rows, using all.")
        return df.reset_index(drop=True)
    return df.sample(n=n, random_state=seed).reset_index(drop=True)


def extract_prompts(df: pd.DataFrame, prompt_col: str = "prompt") -> List[List[dict]]:
    """Extract chat-format message lists from a DataFrame.

    The verl-ready parquet stores prompts as lists of dicts:
        [{"content": "...", "role": "user"}]

    Args:
        df: DataFrame with a prompt column.
        prompt_col: Column name containing chat messages.

    Returns:
        List of chat message lists.
    """
    prompts = []
    for _, row in df.iterrows():
        messages = row[prompt_col]
        if isinstance(messages, str):
            messages = json.loads(messages)
        prompts.append(messages)
    return prompts


def free_model(model, name: str = "model"):
    """Delete model and free GPU/CPU memory."""
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Freed {name}")


In [ ]:
# ============================================================================
# Generation and Log Probability Utilities
# ============================================================================

@torch.no_grad()
def generate_responses(
    model,
    tokenizer,
    prompts: List[List[dict]],
    max_new_tokens: int = 512,
    batch_size: int = 16,
) -> List[dict]:
    """Generate responses using greedy decoding (batch mode).

    Uses left-padding for decoder-only models so that the rightmost
    (most recent) tokens are always aligned.

    Args:
        model: Causal LM on GPU.
        tokenizer: Tokenizer with pad_token set.
        prompts: List of chat message lists.
        max_new_tokens: Maximum new tokens to generate.
        batch_size: Number of prompts per batch.

    Returns:
        List of dicts with keys: prompt_text, response_text,
        prompt_ids (LongTensor), response_ids (LongTensor).
    """
    model.eval()
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    results = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch_msgs = prompts[i : i + batch_size]

        # Apply chat template to get prompt strings
        texts = []
        for msgs in batch_msgs:
            t = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True
            )
            texts.append(t)

        encoded = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
        ).to(model.device)

        # Prompt lengths (excluding left-padding)
        prompt_lengths = encoded.attention_mask.sum(dim=1).tolist()

        output_ids = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

        for j in range(len(batch_msgs)):
            p_len = int(prompt_lengths[j])
            # Account for left-padding offset
            pad_len = encoded.input_ids.shape[1] - p_len
            out = output_ids[j]

            prompt_ids = out[pad_len : pad_len + p_len].cpu()
            response_ids = out[pad_len + p_len :].cpu()
            response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

            results.append({
                "prompt_text": texts[j],
                "response_text": response_text,
                "prompt_ids": prompt_ids,
                "response_ids": response_ids,
            })

    tokenizer.padding_side = original_padding_side
    return results


@torch.no_grad()
def compute_token_log_probs(
    model,
    tokenizer,
    generated_results: List[dict],
) -> List[torch.Tensor]:
    """Compute per-token log probabilities via teacher forcing.

    For each (prompt, response) pair, concatenates prompt_ids and
    response_ids, forward-passes through the model, and extracts
    log p(y_t | y_{<t}, x) at each *response* token position.

    Processes one sample at a time to avoid padding complications.

    Args:
        model: Causal LM on GPU.
        tokenizer: Tokenizer.
        generated_results: Output from generate_responses().

    Returns:
        List of 1-D CPU tensors, one per sample. Each tensor has
        length equal to the number of response tokens.
    """
    model.eval()
    all_log_probs = []

    for result in tqdm(generated_results, desc="Computing log probs"):
        prompt_ids = result["prompt_ids"]
        response_ids = result["response_ids"]

        if len(response_ids) == 0:
            all_log_probs.append(torch.tensor([]))
            continue

        # Concatenate prompt + response
        full_ids = torch.cat([prompt_ids, response_ids]).unsqueeze(0).to(model.device)
        prompt_len = len(prompt_ids)

        logits = model(full_ids).logits  # (1, seq_len, vocab_size)

        # Standard causal LM shift: logits[t] predicts token[t+1]
        shift_logits = logits[:, :-1, :]
        shift_labels = full_ids[:, 1:]

        log_probs = F.log_softmax(shift_logits.float(), dim=-1)
        token_log_probs = log_probs.gather(
            2, shift_labels.unsqueeze(-1)
        ).squeeze(-1)

        # Response tokens start at position prompt_len in the full sequence.
        # After the shift, the log prob for position prompt_len is at index
        # (prompt_len - 1) in token_log_probs.
        resp_start = prompt_len - 1
        resp_lp = token_log_probs[0, resp_start:].cpu().float()

        all_log_probs.append(resp_lp)

    return all_log_probs


def compute_delta_log_p(
    log_probs_ft: List[torch.Tensor],
    log_probs_base: List[torch.Tensor],
) -> List[torch.Tensor]:
    """Compute per-token delta_log_p = log p_finetuned - log p_base.

    Handles length mismatches by truncating to the shorter sequence.

    Args:
        log_probs_ft: Per-sample token log probs from the fine-tuned model.
        log_probs_base: Per-sample token log probs from the base model.

    Returns:
        List of per-token delta_log_p tensors.
    """
    deltas = []
    for lp_ft, lp_base in zip(log_probs_ft, log_probs_base):
        min_len = min(len(lp_ft), len(lp_base))
        if min_len == 0:
            deltas.append(torch.tensor([]))
        else:
            deltas.append(lp_ft[:min_len] - lp_base[:min_len])
    return deltas


In [ ]:
# ============================================================================
# Critical Token Identification
# ============================================================================

def identify_critical_tokens_abs(
    delta_log_p_list: List[torch.Tensor],
    percentile: float = 90,
) -> Tuple[List[torch.Tensor], float]:
    """Identify critical tokens where |delta_log_p| exceeds a percentile threshold.

    This is the basic version used by JWCM v2: selects tokens with the
    largest absolute probability change, regardless of sign.

    Args:
        delta_log_p_list: Per-sample delta_log_p tensors.
        percentile: Percentile threshold (e.g. 90 = top 10% are critical).

    Returns:
        Tuple of (masks, threshold):
        - masks: List of boolean tensors (True = critical token)
        - threshold: Computed absolute threshold value
    """
    # Gather all |delta_log_p| values for global threshold
    all_abs = torch.cat([d.abs() for d in delta_log_p_list if len(d) > 0])
    threshold = torch.quantile(all_abs.float(), percentile / 100.0).item()

    masks = []
    for delta in delta_log_p_list:
        if len(delta) == 0:
            masks.append(torch.tensor([], dtype=torch.bool))
        else:
            masks.append(delta.abs() > threshold)

    total_tokens = sum(len(d) for d in delta_log_p_list)
    critical_count = sum(m.sum().item() for m in masks)
    print(f"|delta_log_p| threshold: {threshold:.4f}")
    print(f"Critical tokens: {critical_count}/{total_tokens} "
          f"({critical_count / max(total_tokens, 1) * 100:.1f}%)")

    return masks, threshold


def identify_critical_tokens_bidirectional(
    delta_log_p_list: List[torch.Tensor],
    percentile: float = 90,
) -> Tuple[List[torch.Tensor], List[torch.Tensor], float, float]:
    """Identify critical tokens with separate amplify (C+) and suppress (C-) sets.

    JWCM v3 uses this to handle RLVR's bidirectional updates:
    - C+ (amplify): tokens whose probability INCREASED (delta_log_p > +threshold)
    - C- (suppress): tokens whose probability DECREASED (delta_log_p < -threshold)

    The threshold is computed from the global distribution of delta_log_p
    (not |delta_log_p|) to allow different thresholds for + and -.

    For simplicity we use symmetric thresholds based on |delta_log_p|.

    Args:
        delta_log_p_list: Per-sample delta_log_p tensors.
        percentile: Percentile for |delta_log_p| threshold.

    Returns:
        Tuple of (masks_plus, masks_minus, threshold_plus, threshold_minus).
    """
    all_abs = torch.cat([d.abs() for d in delta_log_p_list if len(d) > 0])
    threshold = torch.quantile(all_abs.float(), percentile / 100.0).item()

    masks_plus = []
    masks_minus = []
    for delta in delta_log_p_list:
        if len(delta) == 0:
            masks_plus.append(torch.tensor([], dtype=torch.bool))
            masks_minus.append(torch.tensor([], dtype=torch.bool))
        else:
            masks_plus.append(delta > threshold)       # amplified tokens
            masks_minus.append(delta < -threshold)      # suppressed tokens

    total_tokens = sum(len(d) for d in delta_log_p_list)
    plus_count = sum(m.sum().item() for m in masks_plus)
    minus_count = sum(m.sum().item() for m in masks_minus)
    print(f"Threshold: +/-{threshold:.4f}")
    print(f"C+ (amplify):  {plus_count}/{total_tokens} ({plus_count / max(total_tokens, 1) * 100:.1f}%)")
    print(f"C- (suppress): {minus_count}/{total_tokens} ({minus_count / max(total_tokens, 1) * 100:.1f}%)")

    return masks_plus, masks_minus, threshold, threshold


def visualize_delta_log_p(
    delta_log_p_list: List[torch.Tensor],
    task_name: str,
    output_dir: Path,
):
    """Plot distribution of delta_log_p values for a task.

    Creates a histogram showing the distribution of token-level probability
    changes, highlighting the critical regions (tails).

    Args:
        delta_log_p_list: Per-sample delta_log_p tensors.
        task_name: Name for plot title (e.g. "IF" or "Math").
        output_dir: Directory to save the figure.
    """
    all_vals = torch.cat([d for d in delta_log_p_list if len(d) > 0]).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Histogram of delta_log_p
    axes[0].hist(all_vals, bins=200, density=True, alpha=0.7, color="steelblue")
    axes[0].axvline(0, color="red", linestyle="--", linewidth=1)
    axes[0].set_xlabel("delta_log_p")
    axes[0].set_ylabel("Density")
    axes[0].set_title(f"{task_name}: Distribution of delta_log_p")

    # Histogram of |delta_log_p|
    axes[1].hist(np.abs(all_vals), bins=200, density=True, alpha=0.7, color="darkorange")
    p90 = np.percentile(np.abs(all_vals), CRITICAL_PERCENTILE)
    axes[1].axvline(p90, color="red", linestyle="--", linewidth=1,
                    label=f"p{CRITICAL_PERCENTILE} = {p90:.3f}")
    axes[1].set_xlabel("|delta_log_p|")
    axes[1].set_ylabel("Density")
    axes[1].set_title(f"{task_name}: Distribution of |delta_log_p|")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(output_dir / f"delta_log_p_distribution_{task_name.lower()}.png", dpi=150)
    plt.show()
    print(f"Saved {task_name} delta_log_p distribution plot.")


In [ ]:
# ============================================================================
# Task Vector Computation
# ============================================================================

def compute_task_vectors(
    base_model_id: str,
    finetuned_paths: Dict[str, str],
    dtype=torch.float16,
) -> Dict[str, Dict[str, torch.Tensor]]:
    """Compute task vectors: delta_tau = theta_finetuned - theta_base.

    Loads each model pair on CPU, computes the difference per parameter,
    and stores task vectors as CPU tensors in float32 for numerical
    stability during attribution and merging.

    Args:
        base_model_id: HuggingFace ID or path for the base model.
        finetuned_paths: Dict mapping task name -> model path.
        dtype: Load dtype (fp16 for memory efficiency).

    Returns:
        Dict[task_name, Dict[param_name, delta_tensor]].
    """
    print("Loading base model state dict...")
    base_model = load_model(base_model_id, dtype=dtype, device="cpu")
    base_sd = {k: v.float().clone() for k, v in base_model.state_dict().items()}

    task_vectors = {}
    for task_name, ft_path in finetuned_paths.items():
        print(f"Computing task vector for: {task_name}")
        ft_model = load_model(ft_path, dtype=dtype, device="cpu")
        ft_sd = ft_model.state_dict()

        tv = {}
        for name in base_sd:
            if name in ft_sd and ft_sd[name].shape == base_sd[name].shape:
                tv[name] = ft_sd[name].float() - base_sd[name]
            else:
                print(f"  Skipping {name}: not found or shape mismatch")
        task_vectors[task_name] = tv

        del ft_model, ft_sd
        gc.collect()
        print(f"  {task_name}: {len(tv)} parameters, "
              f"total norm = {sum(v.norm().item()**2 for v in tv.values())**0.5:.4f}")

    del base_model, base_sd
    gc.collect()
    return task_vectors


In [ ]:
# ============================================================================
# JWCM v2: Attribution Score Computation
# ============================================================================

def compute_attribution_scores_v2(
    base_model_id: str,
    tokenizer,
    generated_results: List[dict],
    critical_masks: List[torch.Tensor],
    task_vector: Dict[str, torch.Tensor],
    task_name: str,
    dtype=torch.float16,
) -> Dict[str, torch.Tensor]:
    """Compute per-parameter attribution scores S_j for JWCM v2.

    For each validation sample with critical tokens, computes:
        masked_loss = sum of log p(y_t) at critical positions
    Backpropagates through the base model to get gradients, then
    accumulates:
        S_j += |grad_j * delta_theta_j|

    This gives S_j = Sigma_{samples} |grad_j(sample) * delta_theta_j|,
    which approximates the true per-token attribution (with signs
    potentially canceling within each sample's critical set).

    The gradient is computed at theta_base, consistent with the 1st-order
    Taylor expansion: delta_log_p approx nabla log p(theta_base)^T * delta_theta.

    Args:
        base_model_id: Base model path or HF ID.
        tokenizer: Tokenizer.
        generated_results: Generated (prompt, response) pairs.
        critical_masks: Boolean masks over response tokens.
        task_vector: Dict of {param_name: delta_tensor} (float32, CPU).
        task_name: Name for progress bar.
        dtype: Model load dtype.

    Returns:
        Dict[param_name, attribution_tensor] (float32, CPU, same shape as params).
    """
    print(f"\nComputing attribution scores for {task_name}...")
    model = load_model(base_model_id, dtype=dtype, device="cuda")
    model.train()  # Enable gradient computation through all layers

    # Initialize attribution accumulators on CPU (float32)
    S = {}
    for name, param in model.named_parameters():
        param.requires_grad_(True)
        if name in task_vector:
            S[name] = torch.zeros_like(task_vector[name], dtype=torch.float32)

    n_samples_with_critical = 0

    for idx in tqdm(range(len(generated_results)), desc=f"Attribution ({task_name})"):
        mask = critical_masks[idx]
        if len(mask) == 0 or mask.sum().item() == 0:
            continue

        result = generated_results[idx]
        prompt_ids = result["prompt_ids"]
        response_ids = result["response_ids"]

        if len(response_ids) == 0:
            continue

        n_samples_with_critical += 1

        # Concatenate prompt + response and move to GPU
        full_ids = torch.cat([prompt_ids, response_ids]).unsqueeze(0).cuda()
        prompt_len = len(prompt_ids)

        # Forward pass
        logits = model(full_ids).logits

        # Standard causal LM shift
        shift_logits = logits[:, :-1, :]
        shift_labels = full_ids[:, 1:]

        log_probs = F.log_softmax(shift_logits.float(), dim=-1)
        token_log_probs = log_probs.gather(
            2, shift_labels.unsqueeze(-1)
        ).squeeze(-1)

        # Extract response token log probs and apply critical mask
        resp_start = prompt_len - 1
        resp_lp = token_log_probs[0, resp_start:]

        # Truncate mask to match response length
        effective_mask = mask[: len(resp_lp)].float().cuda()
        masked_loss = (resp_lp * effective_mask).sum()

        # Backward pass
        masked_loss.backward()

        # Accumulate |grad * delta_theta| into S on CPU
        for name, param in model.named_parameters():
            if param.grad is not None and name in S:
                grad_cpu = param.grad.detach().float().cpu()
                contrib = (grad_cpu * task_vector[name]).abs()
                S[name] += contrib

        model.zero_grad()

        # Periodic memory cleanup
        if idx % 64 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    print(f"  Processed {n_samples_with_critical} samples with critical tokens")
    total_importance = sum(s.sum().item() for s in S.values())
    print(f"  Total attribution mass: {total_importance:.6f}")

    free_model(model, f"base model (attribution {task_name})")
    return S


In [ ]:
# ============================================================================
# JWCM v2 Merge: Soft Attribution-Weighted Merge
# ============================================================================

ARTIFACT_DIR = Path("merging_analysis/artifacts/jwcm_v2")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MERGED_MODEL_DIR_V2 = ARTIFACT_DIR / "merged_model"
MERGED_MODEL_DIR_V2_SPARSE = ARTIFACT_DIR / "merged_model_sparse"


def merge_jwcm_v2(
    base_model_id: str,
    task_vectors: Dict[str, Dict[str, torch.Tensor]],
    attribution_scores: Dict[str, Dict[str, torch.Tensor]],
    eps: float = 1e-8,
) -> Dict[str, torch.Tensor]:
    """Apply JWCM v2 soft attribution-weighted merge.

    Formula (per parameter j, for N tasks):
        theta_merge_j = theta_base_j
            + sum_tau (S_j^tau * delta_theta_j^tau) / (sum_tau S_j^tau + eps)

    Interpretation: each parameter is updated as a weighted average of
    task vectors, where the weight is the parameter's attribution score
    (how much it contributes to critical token probability changes).

    Args:
        base_model_id: Base model path.
        task_vectors: {task_name: {param_name: delta_tensor}}.
        attribution_scores: {task_name: {param_name: S_tensor}}.
        eps: Small constant to avoid division by zero.

    Returns:
        Merged state dict (all tensors in float32 on CPU).
    """
    print("Applying JWCM v2 merge...")
    base_model = load_model(base_model_id, dtype=DTYPE, device="cpu")
    base_sd = {k: v.float().clone() for k, v in base_model.state_dict().items()}
    del base_model
    gc.collect()

    task_names = list(task_vectors.keys())
    merged_sd = {}
    merge_stats = {"total_params": 0, "merged_params": 0}

    for name in tqdm(base_sd, desc="Merging (v2)"):
        # Check if all tasks have this parameter
        has_all = all(
            name in task_vectors[t] and name in attribution_scores[t]
            for t in task_names
        )

        if not has_all:
            merged_sd[name] = base_sd[name]
            continue

        merge_stats["total_params"] += 1

        # Numerator: sum_tau (S_j^tau * delta_j^tau)
        numerator = torch.zeros_like(base_sd[name])
        denominator = torch.zeros_like(base_sd[name])

        for t in task_names:
            S_t = attribution_scores[t][name]
            delta_t = task_vectors[t][name]
            numerator += S_t * delta_t
            denominator += S_t

        # Weighted merge: base + numerator / (denominator + eps)
        merged_sd[name] = base_sd[name] + numerator / (denominator + eps)
        merge_stats["merged_params"] += 1

    print(f"Merged {merge_stats['merged_params']}/{merge_stats['total_params']} parameters")
    return merged_sd


def merge_jwcm_v2_sparse(
    base_model_id: str,
    task_vectors: Dict[str, Dict[str, torch.Tensor]],
    attribution_scores: Dict[str, Dict[str, torch.Tensor]],
    keep_ratio: float = 0.20,
    eps: float = 1e-8,
) -> Dict[str, torch.Tensor]:
    """Apply JWCM v2 merge with sparsification.

    Same as v2 but adds a global mask: only parameters whose maximum
    attribution score (across tasks) is above a global percentile threshold
    are updated. All other parameters revert to base model values.

    Formula:
        mask_j = 1[max_tau S_j^tau > kappa]
        theta_merge_j = theta_base_j
            + mask_j * sum_tau (S_j^tau * delta_j^tau) / (sum_tau S_j^tau + eps)

    Args:
        base_model_id: Base model path.
        task_vectors: {task_name: {param_name: delta_tensor}}.
        attribution_scores: {task_name: {param_name: S_tensor}}.
        keep_ratio: Fraction of parameters to keep (e.g., 0.20 = top 20%).
        eps: Division epsilon.

    Returns:
        Merged state dict with sparsification applied.
    """
    print(f"Applying JWCM v2 + sparse merge (keep_ratio={keep_ratio})...")
    base_model = load_model(base_model_id, dtype=DTYPE, device="cpu")
    base_sd = {k: v.float().clone() for k, v in base_model.state_dict().items()}
    del base_model
    gc.collect()

    task_names = list(task_vectors.keys())

    # Step 1: Compute global threshold from max attribution across tasks
    # Collect max(S_j^tau) for all mergeable parameters
    all_max_scores = []
    mergeable_params = []
    for name in base_sd:
        has_all = all(
            name in task_vectors[t] and name in attribution_scores[t]
            for t in task_names
        )
        if not has_all:
            continue
        mergeable_params.append(name)
        # max across tasks, per parameter element
        max_S = torch.stack([attribution_scores[t][name] for t in task_names]).max(dim=0).values
        all_max_scores.append(max_S.flatten())

    all_max_scores_cat = torch.cat(all_max_scores)
    # keep_ratio of the parameters: threshold at (1 - keep_ratio) percentile
    kappa = torch.quantile(all_max_scores_cat, 1.0 - keep_ratio).item()
    print(f"  Sparsity threshold kappa: {kappa:.8f}")
    print(f"  Total mergeable scalar params: {len(all_max_scores_cat)}")

    # Step 2: Merge with mask
    merged_sd = {}
    kept_count = 0
    total_count = 0

    for name in tqdm(base_sd, desc="Merging (v2+sparse)"):
        if name not in mergeable_params:
            merged_sd[name] = base_sd[name]
            continue

        max_S = torch.stack([attribution_scores[t][name] for t in task_names]).max(dim=0).values
        mask = (max_S > kappa).float()

        numerator = torch.zeros_like(base_sd[name])
        denominator = torch.zeros_like(base_sd[name])
        for t in task_names:
            S_t = attribution_scores[t][name]
            delta_t = task_vectors[t][name]
            numerator += S_t * delta_t
            denominator += S_t

        delta_merged = numerator / (denominator + eps)
        merged_sd[name] = base_sd[name] + mask * delta_merged

        kept_count += mask.sum().item()
        total_count += mask.numel()

    print(f"  Kept {kept_count}/{total_count} scalar params "
          f"({kept_count / max(total_count, 1) * 100:.1f}%)")
    return merged_sd


def save_merged_model(
    merged_sd: Dict[str, torch.Tensor],
    base_model_id: str,
    output_dir: Path,
    dtype=torch.float16,
):
    """Save merged state dict as a HuggingFace model checkpoint.

    Loads the base model architecture, replaces its weights with the
    merged state dict, and saves in standard HF format.

    Args:
        merged_sd: Merged state dict (float32).
        base_model_id: Base model for architecture.
        output_dir: Directory to save the merged model.
        dtype: Save dtype (fp16 to save disk space).
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Saving merged model to {output_dir}...")
    model = load_model(base_model_id, dtype=torch.float32, device="cpu")

    # Convert merged_sd to the target dtype before loading
    converted_sd = {k: v.to(dtype) for k, v in merged_sd.items()}
    model.load_state_dict(converted_sd, strict=False)

    model.save_pretrained(output_dir)

    # Also copy tokenizer for convenience
    tokenizer = load_tokenizer(base_model_id)
    tokenizer.save_pretrained(output_dir)

    del model, converted_sd
    gc.collect()
    print(f"Saved merged model + tokenizer to {output_dir}")


## Execution: Run the Full Pipeline


In [ ]:
# ============================================================================
# Step 1: Sample Validation Data (512 per benchmark)
# ============================================================================
val_data_dir = ARTIFACT_DIR / "validation_data"
val_data_dir.mkdir(parents=True, exist_ok=True)

val_if_path = val_data_dir / "val_if_512.parquet"
val_math_path = val_data_dir / "val_math_512.parquet"

if val_if_path.exists() and val_math_path.exists():
    print("Loading cached validation data...")
    val_if = pd.read_parquet(val_if_path)
    val_math = pd.read_parquet(val_math_path)
else:
    print("Sampling validation data...")
    val_if = sample_validation_data(IF_DATA_PATH, VAL_SAMPLES_PER_TASK, seed=SEED)
    val_math = sample_validation_data(MATH_DATA_PATH, VAL_SAMPLES_PER_TASK, seed=SEED)
    val_if.to_parquet(val_if_path)
    val_math.to_parquet(val_math_path)

prompts_if = extract_prompts(val_if)
prompts_math = extract_prompts(val_math)
print(f"IF validation samples: {len(prompts_if)}")
print(f"Math validation samples: {len(prompts_math)}")


In [ ]:
# ============================================================================
# Step 2: Generate Responses from Fine-tuned Models
# ============================================================================
gen_dir = ARTIFACT_DIR / "generated_responses"
gen_dir.mkdir(parents=True, exist_ok=True)

tokenizer = load_tokenizer(BASE_MODEL_ID)

# --- Generate IF responses ---
gen_if_path = gen_dir / "gen_if.pt"
if gen_if_path.exists():
    print("Loading cached IF responses...")
    gen_if = torch.load(gen_if_path, weights_only=False)
else:
    print("Generating IF responses...")
    if_model = load_model(IF_MODEL_PATH, dtype=DTYPE, device="cuda")
    gen_if = generate_responses(
        if_model, tokenizer, prompts_if,
        max_new_tokens=MAX_NEW_TOKENS, batch_size=GENERATION_BATCH_SIZE,
    )
    free_model(if_model, "IF model")
    torch.save(gen_if, gen_if_path)
print(f"IF responses: {len(gen_if)}")

# --- Generate Math responses ---
gen_math_path = gen_dir / "gen_math.pt"
if gen_math_path.exists():
    print("Loading cached Math responses...")
    gen_math = torch.load(gen_math_path, weights_only=False)
else:
    print("Generating Math responses...")
    math_model = load_model(MATH_MODEL_PATH, dtype=DTYPE, device="cuda")
    gen_math = generate_responses(
        math_model, tokenizer, prompts_math,
        max_new_tokens=MAX_NEW_TOKENS, batch_size=GENERATION_BATCH_SIZE,
    )
    free_model(math_model, "Math model")
    torch.save(gen_math, gen_math_path)
print(f"Math responses: {len(gen_math)}")

# Print sample responses
for task_name, gen_results in [("IF", gen_if), ("Math", gen_math)]:
    print(f"\n--- {task_name} Sample Response ---")
    r = gen_results[0]
    print(f"Response (first 200 chars): {r['response_text'][:200]}...")
    print(f"Prompt tokens: {len(r['prompt_ids'])}, Response tokens: {len(r['response_ids'])}")


In [ ]:
# ============================================================================
# Step 3: Compute delta_log_p and Identify Critical Tokens
# ============================================================================
dlp_dir = ARTIFACT_DIR / "delta_log_p"
dlp_dir.mkdir(parents=True, exist_ok=True)

dlp_if_path = dlp_dir / "delta_log_p_if.pt"
dlp_math_path = dlp_dir / "delta_log_p_math.pt"

if dlp_if_path.exists() and dlp_math_path.exists():
    print("Loading cached delta_log_p...")
    delta_lp_if = torch.load(dlp_if_path, weights_only=False)
    delta_lp_math = torch.load(dlp_math_path, weights_only=False)
else:
    # Compute log probs under fine-tuned models
    print("Computing log probs under IF model...")
    if_model = load_model(IF_MODEL_PATH, dtype=DTYPE, device="cuda")
    lp_if_ft = compute_token_log_probs(if_model, tokenizer, gen_if)
    free_model(if_model, "IF model (log probs)")

    print("Computing log probs under Math model...")
    math_model = load_model(MATH_MODEL_PATH, dtype=DTYPE, device="cuda")
    lp_math_ft = compute_token_log_probs(math_model, tokenizer, gen_math)
    free_model(math_model, "Math model (log probs)")

    # Compute log probs under base model (for both IF and Math responses)
    print("Computing log probs under Base model...")
    base_model = load_model(BASE_MODEL_ID, dtype=DTYPE, device="cuda")
    lp_if_base = compute_token_log_probs(base_model, tokenizer, gen_if)
    lp_math_base = compute_token_log_probs(base_model, tokenizer, gen_math)
    free_model(base_model, "Base model (log probs)")

    # Compute delta_log_p
    delta_lp_if = compute_delta_log_p(lp_if_ft, lp_if_base)
    delta_lp_math = compute_delta_log_p(lp_math_ft, lp_math_base)

    torch.save(delta_lp_if, dlp_if_path)
    torch.save(delta_lp_math, dlp_math_path)

# Visualize distributions
crit_dir = ARTIFACT_DIR / "critical_token_analysis"
crit_dir.mkdir(parents=True, exist_ok=True)
visualize_delta_log_p(delta_lp_if, "IF", crit_dir)
visualize_delta_log_p(delta_lp_math, "Math", crit_dir)

# Identify critical tokens (v2: |delta_log_p| threshold)
print("\n--- IF Critical Tokens ---")
masks_if, thresh_if = identify_critical_tokens_abs(delta_lp_if, percentile=CRITICAL_PERCENTILE)
print("\n--- Math Critical Tokens ---")
masks_math, thresh_math = identify_critical_tokens_abs(delta_lp_math, percentile=CRITICAL_PERCENTILE)


In [ ]:
# ============================================================================
# Step 4: Compute Task Vectors and Attribution Scores
# ============================================================================
attr_dir = ARTIFACT_DIR / "attribution_scores"
attr_dir.mkdir(parents=True, exist_ok=True)

tv_path = attr_dir / "task_vectors.pt"
attr_if_path = attr_dir / "S_if.pt"
attr_math_path = attr_dir / "S_math.pt"

# Compute task vectors
if tv_path.exists():
    print("Loading cached task vectors...")
    task_vectors = torch.load(tv_path, weights_only=False)
else:
    task_vectors = compute_task_vectors(
        BASE_MODEL_ID,
        {"IF": IF_MODEL_PATH, "Math": MATH_MODEL_PATH},
        dtype=DTYPE,
    )
    torch.save(task_vectors, tv_path)

# Compute attribution scores
if attr_if_path.exists():
    print("Loading cached IF attribution scores...")
    S_if = torch.load(attr_if_path, weights_only=False)
else:
    S_if = compute_attribution_scores_v2(
        BASE_MODEL_ID, tokenizer, gen_if, masks_if,
        task_vectors["IF"], "IF", dtype=DTYPE,
    )
    torch.save(S_if, attr_if_path)

if attr_math_path.exists():
    print("Loading cached Math attribution scores...")
    S_math = torch.load(attr_math_path, weights_only=False)
else:
    S_math = compute_attribution_scores_v2(
        BASE_MODEL_ID, tokenizer, gen_math, masks_math,
        task_vectors["Math"], "Math", dtype=DTYPE,
    )
    torch.save(S_math, attr_math_path)

attribution_scores = {"IF": S_if, "Math": S_math}
print("\nAttribution computation complete.")


In [ ]:
# ============================================================================
# Step 5: Apply JWCM v2 Merge + Sparse Merge + Save
# ============================================================================

# --- v2 (soft attribution merge) ---
merged_sd_v2 = merge_jwcm_v2(
    BASE_MODEL_ID, task_vectors, attribution_scores,
)
save_merged_model(merged_sd_v2, BASE_MODEL_ID, MERGED_MODEL_DIR_V2, dtype=DTYPE)
del merged_sd_v2
gc.collect()

# --- v2 + sparsification ---
merged_sd_v2_sparse = merge_jwcm_v2_sparse(
    BASE_MODEL_ID, task_vectors, attribution_scores,
    keep_ratio=SPARSE_KEEP_RATIO,
)
save_merged_model(merged_sd_v2_sparse, BASE_MODEL_ID, MERGED_MODEL_DIR_V2_SPARSE, dtype=DTYPE)
del merged_sd_v2_sparse
gc.collect()

print("\n" + "=" * 60)
print("JWCM v2 merge complete!")
print(f"  v2 model:        {MERGED_MODEL_DIR_V2}")
print(f"  v2+sparse model: {MERGED_MODEL_DIR_V2_SPARSE}")
print("=" * 60)


## Diagnostics


In [ ]:
# ============================================================================
# Step 6: Diagnostics and Summary
# ============================================================================

def plot_attribution_layer_summary(
    attribution_scores: Dict[str, Dict[str, torch.Tensor]],
    output_dir: Path,
):
    """Visualize per-layer attribution score distribution across tasks.

    For each decoder layer, plots the total attribution mass for each task.
    This reveals which layers are most important for each task.

    Args:
        attribution_scores: {task_name: {param_name: S_tensor}}.
        output_dir: Directory to save figure.
    """
    import re

    task_names = list(attribution_scores.keys())
    layer_scores = {t: defaultdict(float) for t in task_names}

    for t in task_names:
        for name, S in attribution_scores[t].items():
            match = re.search(r"model\.layers\.(\d+)\.", name)
            if match:
                layer_idx = int(match.group(1))
                layer_scores[t][layer_idx] += S.sum().item()
            elif "embed" in name:
                layer_scores[t][-1] += S.sum().item()
            elif "lm_head" in name:
                layer_scores[t][999] += S.sum().item()

    fig, ax = plt.subplots(1, 1, figsize=(14, 6))
    for t in task_names:
        layers = sorted(layer_scores[t].keys())
        scores = [layer_scores[t][l] for l in layers]
        labels = []
        for l in layers:
            if l == -1:
                labels.append("emb")
            elif l == 999:
                labels.append("lm_h")
            else:
                labels.append(str(l))
        ax.plot(range(len(layers)), scores, marker="o", label=t)
        ax.set_xticks(range(len(layers)))
        ax.set_xticklabels(labels, rotation=45)

    ax.set_xlabel("Layer")
    ax.set_ylabel("Total Attribution Score")
    ax.set_title("JWCM: Layer-wise Attribution Score by Task")
    ax.legend()
    ax.set_yscale("log")
    fig.tight_layout()
    fig.savefig(output_dir / "layer_attribution_summary.png", dpi=150)
    plt.show()

plot_attribution_layer_summary(attribution_scores, ARTIFACT_DIR)

# Summary statistics
print("\n--- Attribution Score Summary ---")
for task_name in attribution_scores:
    S = attribution_scores[task_name]
    total = sum(s.sum().item() for s in S.values())
    nonzero_ratio = sum((s > 0).sum().item() for s in S.values()) / sum(s.numel() for s in S.values())
    print(f"  {task_name}: total_mass={total:.4f}, nonzero_ratio={nonzero_ratio:.4f}")
